# 08주차 · 벡터 검색과 RAG

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 수집·분할·임베딩·인덱싱·검색·생성을 분리한다.
- 200자와 500자 청크를 동일한 질문으로 비교한다.
- TF-IDF와 밀집 임베딩 검색의 역할을 구분한다.
- 생성 답변에 근거 문서와 검색 점수를 표시하고 근거가 없으면 응답 보류한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
# 수업용 가상 정책 문서다. 실제 과제에서는 출처와 이용 근거가 있는 원문을 사용한다.
documents = [
    {"id": "P01", "text": "청년 창업 지원 사업은 만 39세 이하 예비 창업자와 창업 3년 이내 기업을 대상으로 한다. 신청자는 사업계획서와 개인정보 동의서를 제출해야 한다. 선정자는 기초 창업 교육과 전문가 상담을 이수한 뒤 사업화 자금을 받을 수 있다. 신청 기간은 8월 1일부터 8월 28일까지이며, 서류가 누락되면 보완 요청을 받을 수 있다. 최종 선정 결과는 심사 종료 후 신청자에게 개별 통지한다."},
    {"id": "P02", "text": "섬 주민 원격진료 지원은 의료기관 방문이 어려운 도서 지역 주민을 위한 상담 사업이다. 주민은 마을 보건 담당자를 통해 상담 날짜를 예약하고 신분 확인 절차를 거친다. 화상 상담에서는 현재 증상과 복용 약을 의료진에게 알린다. 응급 증상이 있거나 대면 검사가 필요하면 원격 상담을 중단하고 가까운 의료기관 방문을 안내한다. 상담 기록은 정해진 보존 기간과 접근 권한에 따라 관리한다."},
    {"id": "P03", "text": "고령자 이동 지원 사업은 병원과 복지관을 방문하기 어려운 이용자에게 예약형 차량을 제공한다. 이용자는 출발 하루 전까지 전화나 복지관 창구에서 승차 위치와 시간을 신청한다. 차량은 지정된 생활권 안에서 운영하며 보호자 동행이 필요한 경우 예약할 때 알려야 한다. 이용 시간 변경이나 취소는 배차 전에 연락해야 한다. 휠체어 사용자는 탑승 장비가 있는 차량인지 미리 확인한다."},
    {"id": "P04", "text": "해양 관광 콘텐츠 지원 사업은 지역의 섬과 항구를 활용한 체험 상품을 개발하는 기업을 돕는다. 지원 분야에는 콘텐츠 기획, 안전 점검, 홍보물 제작과 시범 운영이 포함된다. 신청 기업은 지역 자원 활용 계획과 방문객 안전 대책을 제출해야 한다. 심사에서는 지역성, 실행 가능성, 안전성과 지속 가능성을 함께 평가한다. 단순 장비 구매만을 목적으로 한 신청은 지원 대상에서 제외될 수 있다."},
]

CHUNK_CONDITIONS = {"small": 200, "large": 500}

def chunk_text(text, chunk_size):
    return [text[i:i+chunk_size].strip() for i in range(0, len(text), chunk_size) if text[i:i+chunk_size].strip()]

def make_chunks(documents, condition, chunk_size):
    rows = []
    for doc in documents:
        for order, chunk in enumerate(chunk_text(doc["text"], chunk_size)):
            rows.append({"chunk_id": f"{doc['id']}-{condition}-{order:02d}", "doc_id": doc["id"], "chunk_order": order, "text": chunk})
    return rows

chunk_sets = {name: make_chunks(documents, name, size) for name, size in CHUNK_CONDITIONS.items()}
pd.DataFrame([{"조건": name, "청크크기": CHUNK_CONDITIONS[name], "청크수": len(chunks)} for name, chunks in chunk_sets.items()])


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

questions = [
    {"question_id": "Q01", "question": "청년 창업 지원의 나이 조건은 무엇인가요?", "gold_doc_id": "P01", "answerable": 1},
    {"question_id": "Q02", "question": "섬 주민은 원격 상담 날짜를 어떻게 예약하나요?", "gold_doc_id": "P02", "answerable": 1},
    {"question_id": "Q03", "question": "고령자 차량은 언제까지 신청해야 하나요?", "gold_doc_id": "P03", "answerable": 1},
    {"question_id": "Q04", "question": "교직원 주차요금 감면 기준은 무엇인가요?", "gold_doc_id": None, "answerable": 0},
]

def build_tfidf_index(chunks):
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
    matrix = vectorizer.fit_transform([c["text"] for c in chunks])
    return vectorizer, matrix

def retrieve_tfidf(query, chunks, vectorizer, matrix, k=3):
    scores = cosine_similarity(vectorizer.transform([query]), matrix).ravel()
    order = np.argsort(-scores)[:k]
    return [{**chunks[i], "score": float(scores[i]), "rank": rank} for rank, i in enumerate(order, 1)]

def reciprocal_rank_by_doc(hits, gold_doc_id):
    return next((1 / hit["rank"] for hit in hits if hit["doc_id"] == gold_doc_id), 0.0)

indexes = {name: build_tfidf_index(chunks) for name, chunks in chunk_sets.items()}
metric_rows = []
for condition, chunks in chunk_sets.items():
    vectorizer, matrix = indexes[condition]
    answerable_questions = [q for q in questions if q["answerable"]]
    for q in answerable_questions:
        hits = retrieve_tfidf(q["question"], chunks, vectorizer, matrix)
        rr = reciprocal_rank_by_doc(hits, q["gold_doc_id"])
        metric_rows.append({"condition": condition, "question_id": q["question_id"], "Recall@3": float(rr > 0), "RR": rr})
chunk_metrics = pd.DataFrame(metric_rows)
print(chunk_metrics.to_string(index=False))
summary = chunk_metrics.groupby("condition")[["Recall@3", "RR"]].mean().sort_values(["Recall@3", "RR"], ascending=False)
print(summary.round(3).to_string())
selected_condition = summary.index[0]
selected_chunks = chunk_sets[selected_condition]
selected_vectorizer, selected_matrix = indexes[selected_condition]
print("선택한 청크 조건:", selected_condition, CHUNK_CONDITIONS[selected_condition])


## 밀집 임베딩 검색 비교

아래 모델 셀은 주차별 연습에서는 선택이다. **과제 2에서는 실행하거나 교수자가 제공한 저장 임베딩을 사용하여 TF-IDF와 밀집 임베딩을 반드시 비교한다.**


In [ ]:
RUN_SENTENCE_MODEL = False  # 인터넷 가능한 Colab에서 True
if RUN_SENTENCE_MODEL:
    from sentence_transformers import SentenceTransformer
    dense_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    dense_matrix = dense_model.encode([c["text"] for c in selected_chunks], normalize_embeddings=True)

    def retrieve_dense(query, k=3):
        query_vector = dense_model.encode([query], normalize_embeddings=True)[0]
        scores = dense_matrix @ query_vector
        order = np.argsort(-scores)[:k]
        return [{**selected_chunks[i], "score": float(scores[i]), "rank": rank} for rank, i in enumerate(order, 1)]

    dense_rows = []
    for q in [item for item in questions if item["answerable"]]:
        hits = retrieve_dense(q["question"])
        rr = reciprocal_rank_by_doc(hits, q["gold_doc_id"])
        dense_rows.append({"system": "dense", "question_id": q["question_id"], "Recall@3": float(rr > 0), "RR": rr})
    dense_metrics = pd.DataFrame(dense_rows)
    tfidf_metrics = chunk_metrics.loc[chunk_metrics["condition"] == selected_condition, ["question_id", "Recall@3", "RR"]].assign(system="tfidf")
    system_metrics = pd.concat([tfidf_metrics, dense_metrics], ignore_index=True)
    print(system_metrics.groupby("system")[["Recall@3", "RR"]].mean().round(3).to_string())
    print(pd.DataFrame(retrieve_dense(questions[0]["question"])).to_string(index=False))
else:
    print("Dense 선택 셀을 건너뜁니다. 과제 2에서는 실행 또는 저장 임베딩이 필요합니다.")


In [ ]:
def retrieve_selected_tfidf(query, k=3):
    return retrieve_tfidf(query, selected_chunks, selected_vectorizer, selected_matrix, k=k)

def build_grounded_prompt(question, hits):
    evidence = "\n".join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    return (
        "다음 근거만 사용하여 질문에 답하세요.\n"
        "근거가 부족하면 '제공된 문서로 확인할 수 없습니다'라고 답하세요.\n"
        "문장 끝에 근거의 chunk_id를 표시하세요.\n\n"
        f"[근거]\n{evidence}\n\n[질문]\n{question}\n"
    )

# 네 질문을 작은 개발 데이터로 사용해 답변/응답 보류 점수의 중간값을 고정한다.
dev_score_rows = []
for q in questions:
    top1_score = retrieve_selected_tfidf(q["question"], k=1)[0]["score"]
    dev_score_rows.append({"question_id": q["question_id"], "answerable": q["answerable"], "top1_score": top1_score})
dev_scores = pd.DataFrame(dev_score_rows)
lowest_answerable = dev_scores.loc[dev_scores["answerable"] == 1, "top1_score"].min()
highest_unanswerable = dev_scores.loc[dev_scores["answerable"] == 0, "top1_score"].max()
ABSTAIN_THRESHOLD = float((lowest_answerable + highest_unanswerable) / 2)
print(dev_scores.round(3).to_string(index=False))
print("개발 데이터에서 고정한 응답 보류 임곗값:", round(ABSTAIN_THRESHOLD, 3))

def answer_with_evidence(question, threshold=ABSTAIN_THRESHOLD):
    hits = retrieve_selected_tfidf(question, k=3)
    if not hits or hits[0]["score"] < threshold:
        return {"answer": "제공된 문서로 확인할 수 없습니다.", "action": "abstain", "hits": hits}
    top = hits[0]
    return {"answer": f"{top['text']} [{top['chunk_id']}]", "action": "answer", "hits": hits}

behavior_rows = []
for q in questions:
    result = answer_with_evidence(q["question"])
    expected_action = "answer" if q["answerable"] else "abstain"
    behavior_rows.append({"question_id": q["question_id"], "expected": expected_action, "actual": result["action"], "correct": expected_action == result["action"], "answer": result["answer"]})
behavior_results = pd.DataFrame(behavior_rows)
print(behavior_results.to_string(index=False))
print("행동 정확도:", behavior_results["correct"].mean())
print(build_grounded_prompt(questions[0]["question"], retrieve_selected_tfidf(questions[0]["question"])))


## 학생 활동

- 과제 2와 동일하게 200자·500자, 겹침률 0 조건을 비교하라.
- 주차별 연습에서는 밀집가 선택이지만, 과제 2에서는 밀집 또는 제공된 저장 임베딩을 사용하라.
- 개발 4개에서 청크 크기와 응답 보류 기준을 고정한 뒤 최종 평가 8개를 평가하라.
- 검색 실패와 답변·인용 실패를 별도 열로 기록하라.
- API를 사용한다면 키를 Notebook에 저장하지 말고 환경변수를 사용하라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·응답 보류한 제안, 직접 검증한 내용을 기록하라.
